<a href="https://colab.research.google.com/github/AntonRize/WILL/blob/main/Colab_Notebooks/WILL_RG_Findings_1_2_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WILL RG Galactic Dynamics — reproducible check of Findings 1 and 2
Every number quoted in the conclusions is printed by this notebook.
Inputs: SPARC table1/table2 from github.com/AntonRize/WILL, and the light-bending paper
Mistele et al. 2024 (arXiv:2406.09685v1, ApJL 969 L3), Figure 3 and Section 2.

In [1]:
!pip -q install pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 39.2 MB/s eta 0:00:00


In [2]:
import numpy as np, pandas as pd, urllib.request, re
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy import ndimage
import pypdfium2 as pdfium

# ---------- constants (same values as Galactic_Rotation_Protocol_Independent_SPARC.ipynb)
C_SI=2.99792458e8; G_SI=6.67430e-11; SIGMA_SB=5.670374419e-8
T0_CMB=2.7255; ALPHA=7.2973525693e-3
KPC=3.085677581e19; MPC=3.085677581e22; MSUN=1.98847e30
UPS_DISK=0.5; UPS_BUL=0.7

rho_gamma=4*SIGMA_SB*T0_CMB**4/C_SI**3
rho_max=rho_gamma/(3*ALPHA**2)
H0_SI=np.sqrt(8*np.pi*G_SI*rho_max); H0_KMS=H0_SI*MPC/1e3
a_kappa=C_SI*H0_SI/(3*np.pi); a_beta=C_SI*H0_SI/(6*np.pi); a_Mach=C_SI*H0_SI/(2*np.pi)
print(f"H0 = {H0_KMS:.4f} km/s/Mpc; a_kappa = {a_kappa:.4e}; a_beta = {a_beta:.4e}; a_Mach = {a_Mach:.4e} m/s^2")

H0 = 68.1467 km/s/Mpc; a_kappa = 7.0250e-11; a_beta = 3.5125e-11; a_Mach = 1.0537e-10 m/s^2


In [3]:
base="https://raw.githubusercontent.com/AntonRize/WILL/refs/heads/main/SPARC%20DATA/"
urllib.request.urlretrieve(base+"table1.dat","table1.dat")
urllib.request.urlretrieve(base+"table2.dat","table2.dat")
urllib.request.urlretrieve("https://arxiv.org/pdf/2406.09685v1","mistele2024_v1.pdf")

('mistele2024_v1.pdf', <http.client.HTTPMessage at 0x791703ebb770>)

In [4]:
# ---------- SPARC loading: identical to the document's notebook
cols=[(0,11),(19,25),(26,32),(33,38),(39,45),(46,52),(53,59)]
names_=['Name','Rad','Vobs','e_Vobs','Vgas','Vdisk','Vbul']
df=pd.read_fwf("table2.dat",colspecs=cols,header=None,names=names_,comment='#')
for c in names_[1:]: df[c]=pd.to_numeric(df[c],errors='coerce')
df=df.dropna(subset=['Vobs','Rad']); df=df[(df.Rad>0)&(df.Vobs>0)].copy(); df['Vbul']=df.Vbul.fillna(0.0)
sq=lambda v: np.sign(v)*v**2                       # signed V_gas (outward pull subtracts)
Vb2=sq(df.Vgas.values)+UPS_DISK*sq(df.Vdisk.values)+UPS_BUL*sq(df.Vbul.values)
df['Vb']=np.sqrt(np.maximum(Vb2,0.0))
df['GasFrac']=np.where(Vb2>0, sq(df.Vgas.values)/Vb2, 0.0)
df=df.reset_index(drop=True)
t1=pd.read_fwf("table1.dat",colspecs=[(0,11),(28,29)],names=['Name','fD'],header=None)
df['fD']=df.Name.map(dict(zip(t1.Name.str.strip(),pd.to_numeric(t1.fD,errors='coerce'))))
gas_gal=df.groupby('Name').GasFrac.median(); GAS=set(gas_gal[gas_gal>=0.5].index)
print("points",len(df),"galaxies",df.Name.nunique(),"gas-dominated",len(GAS),
      "| distance origin counts:",df.groupby('Name').fD.first().value_counts().to_dict())

def V_RG(Vb,R,a):                                   # V^2 = V_b^2 + sqrt(V_b^2 a r)
    r=R*KPC; v=Vb*1e3; return np.sqrt(v**2+np.sqrt(v**2*a*r))/1e3
def V_fitted_curve(Vb,R,a0=1.2e-10):                # document's data-fitted curve
    r=R*KPC; gN=(Vb*1e3)**2/r; g=(gN+np.sqrt(gN**2+4*gN*a0))/2; return np.sqrt(g*r)/1e3
def metrics(d,pred):                                # document's metric: per-galaxy median, then median
    t=d.assign(res=d.Vobs-pred).groupby('Name').res
    return dict(MedAE=t.apply(lambda x: np.median(np.abs(x))).median(),
                Bias=t.median().median())           # Bias = V_obs - V_pred : positive = predicts too low

points 3391 galaxies 175 gas-dominated 32 | distance origin counts: {1: 97, 2: 45, 4: 28, 3: 3, 5: 2}


In [5]:
# ---------- Check: reproduce the document's table
for lab,m in [("all",np.ones(len(df),bool)),("gas",df.Name.isin(GAS).values)]:
    d=df[m]
    print(lab,"WILL RG",{k:round(v,2) for k,v in metrics(d,V_RG(d.Vb.values,d.Rad.values,a_kappa)).items()},
          "| data-fitted curve",{k:round(v,2) for k,v in metrics(d,V_fitted_curve(d.Vb.values,d.Rad.values)).items()})

all WILL RG {'MedAE': 11.18, 'Bias': -2.26} | data-fitted curve {'MedAE': 10.43, 'Bias': -4.37}
gas WILL RG {'MedAE': 6.99, 'Bias': 0.53} | data-fitted curve {'MedAE': 7.7, 'Bias': -5.12}


In [6]:
# ---------- FINDING 1: which number does SPARC prefer in a_kappa's slot?
def preferred_per_galaxy(Vb,R,mask):
    out=[]; n_over=0
    for nm in pd.unique(df.Name[mask]):
        s=(df.Name==nm).values
        f=lambda a: np.median(df.Vobs.values[s]-V_RG(Vb[s],R[s],a))
        if f(1e-13)<0: n_over+=1; continue           # over-predicted even with no horizon term
        out.append(brentq(f,1e-13,2e-9,xtol=1e-15))
    return np.array(out)/1e-10, n_over
def zero_bias(Vb,R,mask):
    d=df[mask]; return brentq(lambda a: metrics(d,V_RG(Vb[mask],R[mask],a))['Bias'],0.2e-10,2e-10,xtol=1e-14)/1e-10
grid=np.linspace(0.20e-10,2.00e-10,361)
for corr in (False,True):
    k=np.where((df.fD==1)&corr, 73.0/H0_KMS, 1.0)  # rescale Hubble-flow distances (H0=73) to H0=68.15
    Vb=df.Vb.values*np.sqrt(k); R=df.Rad.values*k   # radius ~ D, V_bar^2 ~ D, V_obs unchanged
    print("\nHubble-flow distances", "rescaled 73 -> %.2f"%H0_KMS if corr else "as tabulated (H0=73)")
    for lab,m in [("all",np.ones(len(df),bool)),("gas",df.Name.isin(GAS).values)]:
        p,nov=preferred_per_galaxy(Vb,R,m); q=np.percentile(p,[25,50,75])
        line=f"  {lab}: zero-bias {zero_bias(Vb,R,m):.3f} | per-galaxy middle {q[1]:.3f}, middle half {q[0]:.2f}-{q[2]:.2f}"
        line+=f" | within 10% of 0.7025: {(abs(p/0.70250-1)<=0.1).sum()} of {len(p)} (over-predicted at zero: {nov})"
        if not corr:
            me=[metrics(df[m],V_RG(Vb[m],R[m],a))['MedAE'] for a in grid]
            line+=f" | smallest typical error at {grid[int(np.argmin(me))]/1e-10:.3f} ({min(me):.2f} km/s)"
            line+=" | typical error at 0.6/0.7/0.8/0.9: "+"/".join(f"{metrics(df[m],V_RG(Vb[m],R[m],a*1e-10))['MedAE']:.2f}" for a in (0.6,0.7,0.8,0.9))
        print(line)


Hubble-flow distances as tabulated (H0=73)
  all: zero-bias 0.602 | per-galaxy middle 0.619, middle half 0.27-1.11 | within 10% of 0.7025: 12 of 170 (over-predicted at zero: 5) | smallest typical error at 0.800 (10.41 km/s) | typical error at 0.6/0.7/0.8/0.9: 11.30/11.16/10.41/10.88
  gas: zero-bias 0.723 | per-galaxy middle 0.722, middle half 0.41-1.49 | within 10% of 0.7025: 5 of 32 (over-predicted at zero: 0) | smallest typical error at 0.830 (6.16 km/s) | typical error at 0.6/0.7/0.8/0.9: 8.01/7.00/6.55/7.43

Hubble-flow distances rescaled 73 -> 68.15
  all: zero-bias 0.539 | per-galaxy middle 0.565, middle half 0.26-1.01 | within 10% of 0.7025: 20 of 167 (over-predicted at zero: 8)
  gas: zero-bias 0.640 | per-galaxy middle 0.655, middle half 0.39-1.30 | within 10% of 0.7025: 3 of 32 (over-predicted at zero: 0)


In [7]:
# ---------- FINDING 2: light-bending speeds far out (Mistele et al. 2024, Figure 3, top-left: all galaxies, 50-300 kpc)
pdf=pdfium.PdfDocument("mistele2024_v1.pdf")
img=pdf[6].render(scale=3).to_pil(); A=np.asarray(img.convert("RGB")).astype(int)
H,W,_=A.shape; s=W/1224
x0,x1,y0,y1=int(300*s),int(662*s),int(135*s),int(556*s)        # top-left panel
dark=A.sum(2)<200
ax_col=x0+np.argmax(dark[y0:y1,x0:x0+80].sum(0))                 # left axis
yt=[y0+i for i in np.where(dark[y0:y1,ax_col+2:ax_col+14].sum(1)>=10)[0]]
ax_row=y0+int(300*s)+np.argmax(dark[y0+int(300*s):y1+20,x0:x1].sum(1))
xt=[ax_col+i for i in np.where(dark[ax_row-14:ax_row-2,ax_col:x1].sum(0)>=8)[0]]
grp=lambda v:[int(np.mean(g)) for g in np.split(np.array(v),np.where(np.diff(v)>3)[0]+1)]
Y11,Y10=[np.mean(g) for g in np.split(np.array(yt),np.where(np.diff(yt)>3)[0]+1)][:2]
xg=[np.mean(g) for g in np.split(np.array(xt),np.where(np.diff(xt)>3)[0]+1)]; X100,X200=xg[-2],xg[-1]
Vof=lambda x:100*10**((x-X100)/((X200-X100)/np.log10(2))); Mof=lambda y:10**(11-(y-Y11)/(Y10-Y11))
P=A[y0:y1,x0:x1]
olive=(P[...,0]>120)&(P[...,0]<200)&(P[...,1]>110)&(P[...,1]<175)&(P[...,2]<80)
lab_,n=ndimage.label(olive); sz=ndimage.sum(olive,lab_,range(1,n+1))
cen=[ndimage.center_of_mass(olive,lab_,i+1) for i in range(n) if 150<sz[i]<400]  # data circles (legend marker is larger)
pts=sorted([(c[1]+x0,c[0]+y0) for c in cen],key=lambda p:p[1])
V_lens=np.array([Vof(x) for x,y in pts]); M73=np.array([Mof(y) for x,y in pts])
print("axis check: 10^11 at y=%.1f, 10^10 at y=%.1f, 100 km/s at x=%.1f, 200 km/s at x=%.1f"%(Y11,Y10,X100,X200))
V_err=np.array([7,9,11,12])   # half-width of horizontal error bars read off the figure (heaviest -> lightest)

corrH=(73.0/H0_KMS)**2        # their baryonic amounts assume H0=73; amount ~ distance^2
M68=M73*corrH
print(f"baryonic amounts rescaled by (73/{H0_KMS:.2f})^2 = {corrH:.4f}")
rows=[]
for V,e,M in zip(V_lens,V_err,M68):
    GM=G_SI*M*MSUN; row={"lensing speed":f"{V:.0f} +/- {e}","baryonic (M_sun, H0=68.15)":f"{M:.3g}"}
    for nm,a in [("2/3: a_kappa",a_kappa),("1/3: a_beta",a_beta),("none: a_Mach",a_Mach)]:
        v=lambda r: np.sqrt(GM/(r*KPC)+np.sqrt(GM*a))/1e3
        row[nm]=f"{v(300):.0f}-{v(50):.0f}"
    rows.append(row)
print(pd.DataFrame(rows[::-1],index=["bin 1 (lightest)","bin 2","bin 3","bin 4 (heaviest)"]).to_string())

axis check: 10^11 at y=385.0, 10^10 at y=720.5, 100 km/s at x=477.0, 200 km/s at x=807.0
baryonic amounts rescaled by (73/68.15)^2 = 1.1475
                 lensing speed baryonic (M_sun, H0=68.15) 2/3: a_kappa 1/3: a_beta none: a_Mach
bin 1 (lightest)    137 +/- 12                   1.46e+10      109-114       92-97      120-125
bin 2               182 +/- 11                    5.3e+10      152-164     128-142      167-178
bin 3                205 +/- 9                   1.04e+11      181-200     153-176      199-217
bin 4 (heaviest)     260 +/- 7                   2.23e+11      221-254     188-227      243-274


In [8]:
# ---------- Distance check: lensing team's redshift->distance shape (matter share 0.2793) vs WILL RG's (1/3, Pantheon+ section)
Dc=lambda z,Om: quad(lambda x:1/np.sqrt(Om*(1+x)**3+1-Om),0,z)[0]
for zl,zs in [(0.15,0.6),(0.25,0.8),(0.35,1.0)]:
    rat=lambda Om: Dc(zs,Om)/(Dc(zs,Om)-Dc(zl,Om))
    v=(rat(1/3)/rat(0.2793))**0.5; m=(Dc(zl,1/3)/Dc(zl,0.2793))**2
    print(f"lens z={zl}, source z={zs}: measured lensing speed x{v:.4f}; predicted speed x{m**0.25:.4f}")

lens z=0.15, source z=0.6: measured lensing speed x1.0031; predicted speed x0.9970
lens z=0.25, source z=0.8: measured lensing speed x1.0049; predicted speed x0.9951
lens z=0.35, source z=1.0: measured lensing speed x1.0065; predicted speed x0.9933
